In [ ]:
%cd /content

# Buat folder output
!mkdir -p /content/output/export_model
!mkdir -p /content/dataset

In [ ]:
# Ekstrak files

!tar -xvzf /content/ssd_mobilenet_v2_fpnlite_320x320.gz -C /content
!unzip -q /content/dataset.zip -d /content/dataset

In [ ]:
# 1. Clone Repositori TensorFlow Models
!git clone --depth 1 https://github.com/tensorflow/models.git

# 2. Compile Protobuf
%cd /content/models/research
!apt-get install -y protobuf-compiler
!protoc object_detection/protos/*.proto --python_out=.

# 3. Install Object Detection API Packages + lvis
!cp object_detection/packages/tf2/setup.py .
!pip install . --use-deprecated=legacy-resolver
!pip install lvis

# 4. Timpa dengan TensorFlow 2.15 & Protobuf Kompatibel (DI AKHIR)
!pip install "tensorflow[and-cuda]==2.15.1" "tf-keras==2.15.0" "tensorflow-estimator==2.15.0" "protobuf==3.20.3" --force-reinstall
!pip install --upgrade cryptography pyOpenSSL tf-slim

In [ ]:

path = "/usr/local/lib/python3.11/dist-packages/tf_slim/data/tfexample_decoder.py"

with open(path, "r") as f:
    content = f.read()

# Perbaiki import dan fungsi legacy control_flow_ops
if "import tensorflow as tf" not in content:
    content = content.replace("from tensorflow.python.ops import control_flow_ops", "import tensorflow as tf\nfrom tensorflow.python.ops import control_flow_ops")

content = content.replace("control_flow_ops.case", "tf.case")
content = content.replace("control_flow_ops.cond", "tf.cond")

with open(path, "w") as f:
    f.write(content)

print("Patching tf_slim berhasil!")

In [ ]:
# Training Model

%cd /content/models/research

!PYTHONPATH=/content/models/research:/content/models/research/slim:/content/models \
TF_USE_LEGACY_KERAS=1 \
PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python \
python object_detection/model_main_tf2.py \
    --pipeline_config_path=/content/pipeline.config \
    --model_dir=/content/output \
    --alsologtostderr

In [ ]:
# Export Graph Model

%cd /content/models/research

!PYTHONPATH=/content/models/research:/content/models/research/slim:/content/models \
TF_USE_LEGACY_KERAS=1 \
PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python \
python object_detection/export_tflite_graph_tf2.py \
    --pipeline_config_path=/content/pipeline.config \
    --trained_checkpoint_dir=/content/output \
    --output_directory=/content/output/export_model \
    --max_detections=10

In [ ]:
# Konversi ke TFLite

import tensorflow as tf

saved_model_dir = "/content/output/export_model/saved_model"
output_tflite = "/content/ssdmobilnet-fpnlite.tflite"

print("TensorFlow:", tf.__version__)

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

tflite_model = converter.convert()

with open(output_tflite, "wb") as f:
    f.write(tflite_model)

print("Saved:", output_tflite)